Bloco 1 — Instalar e importar bibliotecas

In [ ]:
import pandas as pd
import re

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


Bloco 2 — Carregar o arquivo

In [ ]:
df = pd.read_excel('extrato_conta_corrente.xlsx', header=4, sheet_name='Sheet')
print(f"Shape: {df.shape}")
print(df.head(3))

Shape: (2457, 9)
  Lancto        Data Bloco     Conta  \
0    NaN         NaN   NaN       NaN   
1  00000  19/09/2025     1  00002453   
2  00000  19/09/2025     1  00002453   

                                           Histórico  Unnamed: 5  Débito  \
0                                     SALDO ANTERIOR         NaN     NaN   
1  Recibo nº: 22748315 - Unidade: 000303 - Vencto...         NaN     NaN   
2  Recibo nº: 22748349 - Unidade: 000601 - Vencto...         NaN     NaN   

  Unnamed: 7  Crédito  
0        NaN      NaN  
1        NaN   364.40  
2        NaN   366.02  


Bloco 3 — Remover colunas desnecessárias

In [ ]:
df = df.drop(columns=['Unnamed: 5', 'Unnamed: 7'])
print(df.columns.tolist())

['Lancto', 'Data', 'Bloco', 'Conta', 'Histórico', 'Débito', 'Crédito']


Bloco 4 — Remover linhas inválidas

In [ ]:
df = df[df['Data'].notna()]
df = df[df['Bloco'].isin([0, 1, '0', '1'])]
df = df.reset_index(drop=True)
print(f"Shape após limpeza: {df.shape}")

Shape após limpeza: (2449, 7)


Bloco 5 — Converter Data

In [ ]:
df['Data'] = pd.to_datetime(df['Data'], dayfirst=True, errors='coerce')
df = df[df['Data'].notna()].reset_index(drop=True)
print(df['Data'].dtype)
print(f"Shape após conversão: {df.shape}")
print(df['Data'].head())

datetime64[ns]
Shape após conversão: (2448, 7)
0   2025-09-19
1   2025-09-19
2   2025-09-19
3   2025-09-22
4   2025-09-22
Name: Data, dtype: datetime64[ns]


Bloco 6 — Criar coluna Tipo e Valor

In [ ]:
df['Tipo'] = df['Crédito'].apply(lambda x: 'C' if pd.notna(x) else 'D')
df['Valor'] = df.apply(
    lambda row: row['Crédito'] if pd.notna(row['Crédito']) else row['Débito'],
    axis=1
)
df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
print(df[['Tipo', 'Valor']].head(10))

  Tipo   Valor
0    C  364.40
1    C  366.02
2    C  364.40
3    C  364.40
4    C  366.02
5    C  366.02
6    C  364.40
7    C  359.76
8    C  364.40
9    C  310.68


Bloco 7 — Extrair Unidade

In [ ]:
def extrair_favorecido_credito(row):
    h = str(row['Histórico']).strip()
    if row['Tipo'] != 'C':
        return None
    m = re.search(r'[Uu]nidade[:\s]+(\d+)', h)
    if m:
        return 'Unidade ' + m.group(1)
    m = re.search(r'UNIDADE[:\s]+(\d+)', h)
    if m:
        return 'Unidade ' + m.group(1)
    return None

df['Favorecido'] = df.apply(extrair_favorecido_credito, axis=1)
print(df[df['Tipo']=='C'][['Histórico','Favorecido']].head(10))
print(f"\nNulos nos créditos: {df[df['Tipo']=='C']['Favorecido'].isna().sum()}")
print(f"Unidades únicas: {df['Favorecido'].nunique()}")

                                           Histórico      Favorecido
0  Recibo nº: 22748315 - Unidade: 000303 - Vencto...  Unidade 000303
1  Recibo nº: 22748349 - Unidade: 000601 - Vencto...  Unidade 000601
2  Recibo nº: 22748411 - Unidade: 001103 - Vencto...  Unidade 001103
3  Recibo nº: 22748298 - Unidade: 000110 - Vencto...  Unidade 000110
4  Recibo nº: 22748300 - Unidade: 000112 - Vencto...  Unidade 000112
5  Recibo nº: 22748313 - Unidade: 000301 - Vencto...  Unidade 000301
6  Recibo nº: 22748339 - Unidade: 000503 - Vencto...  Unidade 000503
7  Recibo nº: 22748398 - Unidade: 001002 - Vencto...  Unidade 001002
8  Recibo nº: 22748399 - Unidade: 001003 - Vencto...  Unidade 001003
9  Recibo nº: 22748404 - Unidade: 001008 - Vencto...  Unidade 001008

Nulos nos créditos: 9
Unidades únicas: 332


Bloco 8 — Extrair Num_Recibo e Data_Vencimento

In [ ]:
def extrair_recibo(h):
    m = re.search(r'Recibo n[ºo][\.:]\s*(\d+)', str(h), re.IGNORECASE)
    return m.group(1) if m else None

def extrair_vencimento(h):
    m = re.search(r'Vencto[:\s]+(\d{2}/\d{2}/\d{4})', str(h), re.IGNORECASE)
    if m:
        try:
            return pd.to_datetime(m.group(1), dayfirst=True)
        except:
            return None
    return None

df['Num_Recibo'] = df['Histórico'].apply(extrair_recibo)
df['Data_Vencimento'] = df['Histórico'].apply(extrair_vencimento)
print(df[['Num_Recibo', 'Data_Vencimento']].head(10))

  Num_Recibo Data_Vencimento
0   22748315      2025-09-25
1   22748349      2025-09-25
2   22748411      2025-09-25
3   22748298      2025-09-25
4   22748300      2025-09-25
5   22748313      2025-09-25
6   22748339      2025-09-25
7   22748398      2025-09-25
8   22748399      2025-09-25
9   22748404      2025-09-25


Bloco 9 — Criar Favorecido_Padronizado e Categoria

In [ ]:
# ================================
# MAPEAMENTO DE PADRONIZAÇÃO
# ================================
mapeamento = [
    (r'WINKER',                         'Winker',                    'Tecnologia'),
    (r'DCG',                            'DCG Facilities',            'Pessoal Terceirizado'),
    (r'SABESP',                         'Sabesp',                    'Utilities'),
    (r'ENEL',                           'Enel',                      'Utilities'),
    (r'PLAZA SOLUTIONS',                'Plaza Solutions',           'Administração'),
    (r'BEN LIMPEZA',                    'Ben Limpeza de Piscinas',   'Manutenção'),
    (r'ATLAS SCHINDLER|ELEVADORES ATLAS','Elevadores Atlas Schindler','Manutenção'),
    (r'RB TECH',                        'RB Tech Engenharia',        'Manutenção'),
    (r'TOKIO MARINE|TOKIO',             'Tokio Marine',              'Seguros'),
    (r'VEMA',                           'Vema Comércio',             'Materiais'),
    (r'INSS',                           'INSS',                      'Tributos'),
    (r'CSLL|COFINS|PIS',                'CSLL/COFINS/PIS',           'Tributos'),
    (r'TAXA ADMINISTRA',                'Taxa Administração',        'Administração'),
    (r'JD SEGU',                        'JD Segurança',              'Segurança'),
    (r'INSET PRAGAS',                   'Inset Pragas',              'Manutenção'),
    (r'SIGMA EXTINTORES',               'Sigma Extintores',          'Segurança'),
    (r'MERCADOPAGO',                    'Mercado Pago',              'Materiais'),
    (r'PAGAR\.ME',                      'Pagar.me',                  'Materiais'),
    (r'YAPAY',                          'Yapay / Microsoft',         'Tecnologia'),
    (r'GSVK',                           'GSVK Caçambas',             'Serviços Gerais'),
    (r'SAO VICENTE',                    'São Vicente Fomento',       'Materiais'),
    (r'OMEGA PREDIOS',                  'Omega Prédios',             'Materiais'),
    (r'TARIFA|TAR/CUSTAS',              'Tarifas Bancárias',         'Administrativo'),
    (r'TRANSFER|TED|PIX',               'Transferência',             'Administrativo'),
    (r'EXPERT FITNESS',                 'Expert Fitness',            'Manutenção'),
    (r'LEROY MERLIN',                   'Leroy Merlin',              'Materiais'),
    (r'MADEIRA MADEIRA',                'Madeira Madeira',           'Materiais'),
    (r'MAGALUPAY',                      'Magalupay',                 'Materiais'),
    (r'KALUNGA',                        'Kalunga',                   'Materiais'),
    (r'NELSON FERNANDES',               'Nelson Fernandes Vieira',   'Serviços Gerais'),
    (r'EDSON.*PARDINHO',                'Edson Pardinho',            'Serviços Gerais'),
    (r'EUNERIO',                        'Eunerio Jose Leite',        'Serviços Gerais'),
    (r'CERTSIM',                        'Certsim',                   'Administrativo'),
    (r'CARTÓRIO|CARTORIO',              'Cartório',                  'Administrativo'),
    (r'IMPRESS',                        'Impressões',                'Administrativo'),
    (r'R2A REFORMAS|R2A',               'R2A Reformas',              'Manutenção'),
    (r'ACERTO CONTABIL',                'Acerto Contábil',           'Administrativo'),
    (r'GURGELMIX',                      'Gurgelmix',                 'Materiais'),
    (r'ESOCIAL',                        'eSocial',                   'Administrativo'),
    (r'GESTAO DE RECEBIMENTOS',         'Gestão de Recebimentos',    'Administrativo'),
    (r'RETEN',                          'Gerenciamento Tributos',    'Tributos'),
    (r'CERTIFICACAO DIGITAL',           'Certificação Digital',      'Administrativo'),
    (r'MONITORAMENTO JUDICIAL',         'Monitoramento Judicial',    'Administrativo'),
    (r'CONFECCAO DE PASTAS',            'Confecção de Pastas',       'Administrativo'),
    (r'REFRIGERADOR',                   'Compra de Equipamento',     'Materiais'),
    (r'ROBOTTON',                       'Robotton Master',           'Administração'),

    # Condôminos (entrada de receita)
    (r'RECIBO|UNIDADE|APTO|COND',       None,                        'Condômino'),
]

# ================================
# FUNÇÃO DE PADRONIZAÇÃO
# ================================
def padronizar(row):
    h = str(row['Histórico']).upper().strip()

    for padrao, nome, categoria in mapeamento:
        if re.search(padrao, h, re.IGNORECASE):

            # Condômino → mantém histórico como favorecido
            if nome is None:
                return h, categoria

            return nome, categoria

    # Caso não encontre
    return h, 'Outros'

# ================================
# APLICAR NO DATAFRAME
# ================================
df[['Favorecido_Padronizado', 'Categoria']] = df.apply(
    padronizar,
    axis=1,
    result_type='expand'
)

# ================================
# VALIDAÇÃO
# ================================
print(df[['Histórico', 'Favorecido_Padronizado', 'Categoria']].head(20))

                                            Histórico  \
0   Recibo nº: 22748315 - Unidade: 000303 - Vencto...   
1   Recibo nº: 22748349 - Unidade: 000601 - Vencto...   
2   Recibo nº: 22748411 - Unidade: 001103 - Vencto...   
3   Recibo nº: 22748298 - Unidade: 000110 - Vencto...   
4   Recibo nº: 22748300 - Unidade: 000112 - Vencto...   
5   Recibo nº: 22748313 - Unidade: 000301 - Vencto...   
6   Recibo nº: 22748339 - Unidade: 000503 - Vencto...   
7   Recibo nº: 22748398 - Unidade: 001002 - Vencto...   
8   Recibo nº: 22748399 - Unidade: 001003 - Vencto...   
9   Recibo nº: 22748404 - Unidade: 001008 - Vencto...   
10  Recibo nº: 22748405 - Unidade: 001009 - Vencto...   
11  Recibo nº: 22748444 - Unidade: 001312 - Vencto...   
12  Recibo nº: 22748446 - Unidade: 001402 - Vencto...   
13  Recibo nº: 22748447 - Unidade: 001403 - Vencto...   
14  Recibo nº: 22748453 - Unidade: 001409 - Vencto...   
15  Recibo nº: 22748455 - Unidade: 001411 - Vencto...   
16  Recibo nº: 22748477 - Unida

Bloco 10 — Extrair descrição para registros de juros de condomínio

In [ ]:
mask_juros = df['Histórico'].str.contains(
    'RECEBIMENTO DE JUROS CONDOMINIO',
    case=False,
    na=False
)

df.loc[mask_juros, 'Favorecido_Padronizado'] = 'Juros'
df.loc[mask_juros, 'Categoria'] = 'Condômino'

# Validação
df[mask_juros][['Histórico', 'Favorecido_Padronizado', 'Categoria']].head()

,Histórico,Favorecido_Padronizado,Categoria
190,RECEBIMENTO DE JUROS CONDOMINIO: 912 - BLOCO: ...,Juros,Condômino
191,RECEBIMENTO DE JUROS CONDOMINIO: 912 - BLOCO: ...,Juros,Condômino
194,RECEBIMENTO DE JUROS CONDOMINIO: 912 - BLOCO: ...,Juros,Condômino
195,RECEBIMENTO DE JUROS CONDOMINIO: 912 - BLOCO: ...,Juros,Condômino
196,RECEBIMENTO DE JUROS CONDOMINIO: 912 - BLOCO: ...,Juros,Condômino


Bloco 11 - Padronização texto justificado : Unidade

In [ ]:
import re

def padronizar_unidade(texto):
    if pd.isna(texto):
        return texto

    texto = str(texto).strip().upper()

    match = re.search(r'UNIDADE\s*(\d+)', texto)
    if match:
        numero = match.group(1)
        return f'Unidade {numero}'

    return texto

# Aplicar na coluna
df['Favorecido_Padronizado'] = df['Favorecido_Padronizado'].apply(padronizar_unidade)

# Validação
df[df['Favorecido_Padronizado'].str.contains('Unidade', na=False)].head()

,Lancto,Data,Bloco,Conta,Histórico,Débito,Crédito,Tipo,Valor,Favorecido,Num_Recibo,Data_Vencimento,Favorecido_Padronizado,Categoria


Bloco 12 - Remover coluna Favorecido

In [ ]:
df = df.drop(columns=['Favorecido'], errors='ignore')

# Validação
print(df.columns)

Index(['Lancto', 'Data', 'Bloco', 'Conta', 'Histórico', 'Débito', 'Crédito',
       'Tipo', 'Valor', 'Num_Recibo', 'Data_Vencimento',
       'Favorecido_Padronizado', 'Categoria'],
      dtype='object')


Bloco 13 — Limpeza e Organização da Tabela (Fact)

In [ ]:
# Remover colunas desnecessárias
colunas_remover = [
    'Débito',
    'Crédito',
    'Favorecido'
]

df = df.drop(columns=colunas_remover, errors='ignore')

# Reorganizar colunas (mantendo Histórico)
ordem_colunas = [
    'Data',
    'Data_Vencimento',
    'Lancto',
    'Bloco',
    'Conta',
    'Tipo',
    'Histórico',
    'Categoria',
    'Favorecido_Padronizado',
    'Valor',
    'Num_Recibo'
]

df = df[ordem_colunas]

# Renomear coluna final
df = df.rename(columns={
    'Favorecido_Padronizado': 'Favorecido'
})

# Ordenar dados
df = df.sort_values(by=['Data', 'Lancto'])

# Reset index
df = df.reset_index(drop=True)

# Validação
print(df.head())
print(df.info())

        Data Data_Vencimento Lancto Bloco     Conta Tipo  \
0 2025-09-19      2025-09-25  00000     1  00002453    C   
1 2025-09-19      2025-09-25  00000     1  00002453    C   
2 2025-09-19      2025-09-25  00000     1  00002453    C   
3 2025-09-22      2025-09-25  00000     1  00002453    C   
4 2025-09-22      2025-09-25  00000     1  00002453    C   

                                           Histórico  Categoria  \
0  Recibo nº: 22748315 - Unidade: 000303 - Vencto...  Condômino   
1  Recibo nº: 22748349 - Unidade: 000601 - Vencto...  Condômino   
2  Recibo nº: 22748411 - Unidade: 001103 - Vencto...  Condômino   
3  Recibo nº: 22748298 - Unidade: 000110 - Vencto...  Condômino   
4  Recibo nº: 22748300 - Unidade: 000112 - Vencto...  Condômino   

                                          Favorecido   Valor Num_Recibo  
0  RECIBO Nº: 22748315 - UNIDADE: 000303 - VENCTO...  364.40   22748315  
1  RECIBO Nº: 22748349 - UNIDADE: 000601 - VENCTO...  366.02   22748349  
2  RECIBO Nº: 

Bloco 14 - Limpeza e ajustes finais

In [ ]:
# Num_Recibo está como float — 22748315.0 em vez de 22748315. Corrigir para inteiro ou texto:
df['Num_Recibo'] = df['Num_Recibo'].fillna(0).astype(int).replace(0, None)

# Dois registros na categoria Outros — vale ver o que são antes de fechar
print(df[df['Categoria'] == 'Outros'][['Histórico', 'Favorecido', 'Categoria']].to_string())

                                                                Histórico                                                           Favorecido Categoria
758                         GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS    Outros
1034                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS    Outros
1352                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS    Outros
1356                                        CONFECÇÃO DE PASTAS PC. 02/02                                        CONFECÇÃO DE PASTAS PC. 02/02    Outros
1714                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS    Outros
2068                        GESTÃO DE RECEBIMENTOS/CONTROLE DE PENDÊNCIAS         

Bloco 15 - Gerar dim_Favorecido com chave surrogate

In [ ]:
dim_favorecido = df[['Favorecido', 'Categoria']]\
    .drop_duplicates()\
    .sort_values('Favorecido')\
    .reset_index(drop=True)

dim_favorecido.insert(0, 'Favorecido_Key', range(1, len(dim_favorecido) + 1))

print(f"Total favorecidos únicos: {len(dim_favorecido)}")
print(dim_favorecido.head(10).to_string())

Total favorecidos únicos: 1957
   Favorecido_Key                                                           Favorecido       Categoria
0               1                                                      ACERTO CONTÁBIL  Administrativo
1               2  ARTE DI FIORI PAISAGISMO E DEC LTDA - NF. 4458. - MANUTENÇÃO JARDIM          Outros
2               3  ARTE DI FIORI PAISAGISMO E DEC LTDA - NF. 4459. - MANUTENÇAO JARDIM          Outros
3               4  ARTE DI FIORI PAISAGISMO E DEC LTDA - NF. 4460. - MANUTENÇÃO JARDIM          Outros
4               5                                              BEN LIMPEZA DE PISCINAS      Manutenção
5               6                                                             CARTÓRIO  Administrativo
6               7                                                              CERTSIM  Administrativo
7               8                                                COMPRA DE EQUIPAMENTO       Materiais
8               9                         

Bloco 16 - Registrar Favorecido_Key na tabela fato

In [ ]:
df = df.merge(dim_favorecido[['Favorecido_Key', 'Favorecido']],
              on='Favorecido',
              how='left')

df = df.drop(columns=['Favorecido', 'Categoria'])

print(f"Shape final: {df.shape}")
print(df.columns.tolist())
print(df[['Favorecido_Key']].value_counts().head(10))

Shape final: (2448, 10)
['Data', 'Data_Vencimento', 'Lancto', 'Bloco', 'Conta', 'Tipo', 'Histórico', 'Valor', 'Num_Recibo', 'Favorecido_Key']
Favorecido_Key
29                354
1951               20
11                 14
14                 11
1955               11
5                   9
1956                7
1952                7
1948                7
40                  6
Name: count, dtype: int64


In [ ]:
# Criar cópia para validação
df_temp = df.copy()

# Exportar para Excel
df_temp.to_excel('tmp_f_Conta_Corrente.xlsx', index=False)

print("Arquivo de validação gerado com sucesso!")

Arquivo de validação gerado com sucesso!


In [ ]:
# Criar cópia para validação
dfav_temp = dim_favorecido.copy()

# Exportar para Excel
dfav_temp.to_excel('tmp_dim_Favorecido.xlsx', index=False)

print("Arquivo de validação gerado com sucesso!")

Arquivo de validação gerado com sucesso!
